In [21]:
from src.load import load_orders, load_order_products, load_products

orders = load_orders()
order_products = load_order_products()
products = load_products()

In [22]:
import pandas as pd
pd.set_option("display.float_format", "{:.2f}".format)

In [23]:
from src.load import load_orders, load_order_products

orders = load_orders()
print(orders.shape)
print(orders.memory_usage(deep=True).sum() / 1e6, "MB")
orders.head()

(3421083, 7)
58.158836 MB


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.00
2,473747,1,prior,3,3,12,21.00
3,2254736,1,prior,4,4,7,29.00
4,431534,1,prior,5,4,15,28.00


In [24]:
from src.load import load_order_products

order_products = load_order_products()
print(order_products.shape)

(32434489, 4)


In [25]:
print(order_products.memory_usage(deep=True).sum() / 1e6, "MB")


356.779511 MB


In [26]:
basket_size = order_products.groupby("order_id").size()
print(basket_size.describe())

count   3214874.00
mean         10.09
std           7.53
min           1.00
25%           5.00
50%           8.00
75%          14.00
max         145.00
dtype: float64


In [27]:
print(orders["eval_set"].value_counts())

eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64


In [28]:
print(orders["user_id"].nunique())

206209


In [29]:
prior_orders = orders.loc[
    orders["eval_set"] == "prior",
    ["order_id", "user_id", "order_number"]
]

lines = order_products.merge(prior_orders, on="order_id", how="inner")

In [30]:
print(lines.shape)
print(lines.memory_usage(deep=True).sum() / 1e6, "MB")
lines.head()

(32434489, 6)
551.386445 MB


,order_id,product_id,add_to_cart_order,reordered,user_id,order_number
0,2,33120,1,1,202279,3
1,2,28985,2,1,202279,3
2,2,9327,3,0,202279,3
3,2,45918,4,1,202279,3
4,2,30035,5,0,202279,3


In [31]:
del order_products

In [32]:
print(orders["days_since_prior_order"].median())

user_orders = prior_orders.groupby("user_id").size()

print(user_orders.describe())
print((user_orders >= 12).mean())

7.0
count   206209.00
mean        15.59
std         16.65
min          3.00
25%          5.00
50%          9.00
75%         19.00
max         99.00
dtype: float64
0.42155773996285323


In [33]:
orders["days_since_prior_order"].max()

np.float32(30.0)

In [34]:
lines_per_user = lines.groupby("user_id").size()

for n in [6, 8, 10, 12]:
    keep = user_orders >= n
    users_kept = keep.mean()
    lines_kept = lines_per_user[keep].sum() / len(lines)
    print(n, round(users_kept, 2), round(lines_kept, 2))

6 0.71 0.93
8 0.59 0.88
10 0.49 0.83
12 0.42 0.78


In [35]:
MIN_ORDERS = 12

up = lines.groupby(["user_id", "product_id"]).size().rename("times_bought").reset_index()
up["n_orders"] = up["user_id"].map(user_orders)
up = up[up["n_orders"] >= MIN_ORDERS]
up["share"] = up["times_bought"] / up["n_orders"]

for t in [0.25, 0.5]:
    reg = up[up["share"] >= t]
    print(t, len(reg), reg.groupby("user_id").size().median())

0.25 822287 8.0
0.5 223366 2.0


In [36]:
eligible = user_orders[user_orders >= MIN_ORDERS].index.to_series()
sample_users = eligible.sample(2000, random_state=42)

lines_s = lines[lines["user_id"].isin(sample_users)]
print(len(lines_s))

for SHARE in [0.25, 0.5]:
    reg_s = up[(up["share"] >= SHARE) & (up["user_id"].isin(sample_users))]
    print(SHARE, len(reg_s))

596948
0.25 19609
0.5 5286


In [37]:
SHARE = 0.5
reg_s = up[(up["share"] >= SHARE) & (up["user_id"].isin(sample_users))][["user_id", "product_id"]]

orders_s = lines_s[["user_id", "order_id"]].drop_duplicates()
expected = orders_s.merge(reg_s, on="user_id")

bought = lines_s[["order_id", "product_id"]].assign(present=1)
expected = expected.merge(bought, on=["order_id", "product_id"], how="left")

missing = expected[expected["present"].isna()]

print(len(expected), len(missing), round(len(missing) / len(expected), 2))

136321 47904 0.35


In [38]:
aisle = products.set_index("product_id")["aisle_id"]

missing = missing.assign(aisle_id=missing["product_id"].map(aisle))

aisles_in_order = (
    lines_s.assign(aisle_id=lines_s["product_id"].map(aisle))[["order_id", "aisle_id"]]
    .drop_duplicates()
    .assign(aisle_present=1)
)

checked = missing.merge(aisles_in_order, on=["order_id", "aisle_id"], how="left")

print(len(checked), round(checked["aisle_present"].notna().mean(), 2))

47904 0.36


In [39]:
present = expected[expected["present"].notna()].copy()
present["aisle_id"] = present["product_id"].map(aisle)

aisle_counts = (
    lines_s.assign(aisle_id=lines_s["product_id"].map(aisle))
    .groupby(["order_id", "aisle_id"]).size()
    .rename("n_in_aisle")
    .reset_index()
)

present = present.merge(aisle_counts, on=["order_id", "aisle_id"], how="left")

print(len(present), round((present["n_in_aisle"] >= 2).mean(), 2))

88417 0.45


In [40]:
present = present.merge(
    lines_s[["order_id", "product_id", "reordered"]],
    on=["order_id", "product_id"], how="left"
)
present = present[present["reordered"] == 1]

new_in_aisle = (
    lines_s[lines_s["reordered"] == 0]
    .assign(aisle_id=lambda d: d["product_id"].map(aisle))[["order_id", "aisle_id"]]
    .drop_duplicates()
    .assign(new_present=1)
)

for name, df in [("missing", missing), ("present", present)]:
    m = df.merge(new_in_aisle, on=["order_id", "aisle_id"], how="left")
    print(name, len(m), round(m["new_present"].notna().mean(), 3))

missing 47904 0.131
present 83131 0.106


In [42]:
print(len(lines_s), len(missing), len(present))

596948 47904 83131


In [43]:
new_lines = lines_s.loc[lines_s["reordered"] == 0, ["order_id", "product_id"]]
new_lines = new_lines.rename(columns={"product_id": "b"})

def count_events(events):
    e = events[["order_id", "product_id"]].rename(columns={"product_id": "a"})
    n = e.groupby("a").size()
    k = e.merge(new_lines, on="order_id").groupby(["a", "b"]).size()
    return n, k

n_m, k_m = count_events(missing)
n_b, k_b = count_events(present)

print(len(n_m), len(k_m))
print(len(n_b), len(k_b))

2296 182632
2316 242753


In [44]:
print(k_m.value_counts().sort_index().head(6))
print((k_m >= 5).sum())

1    171595
2      7462
3      1824
4       694
5       372
6       221
Name: count, dtype: int64
1057
